### Initialisation

Doing all the relevant imports

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

# install dependencies
!pip install sympy==1.13.1
!pip install grad-cam --quiet

#do required imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # used for showing all images in a grid
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.utils.class_weight import compute_class_weight

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from scipy.optimize import linear_sum_assignment  # for cosine similarity
import shutil  #??

# verify gpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

### Handling HAM10000 and Derm7pt data and repos

This code takes the HAM10000 dataset, Derm7pt dataset, and ICE repo and saves them to drive if not already done. Otherwise it simply loads them.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/ham10000_dataset/dataset/ham10000'
DATA_DIR  = '/content/dataset/ham10000'
REPO_DIR  = '/content/InvertibleCE'
HAM_OUTPUT_DIR = '/content/drive/MyDrive/ham10000_dataset/outputs'

os.makedirs(HAM_OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print('Cloning repo and installing dependencies...')
    !git clone https://github.com/zhangrh93/InvertibleCE {REPO_DIR}
    os.chdir(REPO_DIR)
    !sed -i 's/torch==.*/torch/' requirements.txt
    !sed -i 's/torchvision==.*/torchvision/' requirements.txt
    !pip install -r requirements.txt
    print('Done.')
else:
    print('Repo already cloned, skipping.')
    os.chdir(REPO_DIR)

EXPLAINERS_DRIVE_DIR = os.path.join(HAM_OUTPUT_DIR, 'Explainers')
os.makedirs(EXPLAINERS_DRIVE_DIR, exist_ok=True)

local_explainers_path = os.path.join(REPO_DIR, 'Explainers')
if os.path.islink(local_explainers_path):
    print('Explainers symlink already exists, skipping.')
elif os.path.exists(local_explainers_path):
    for item in os.listdir(local_explainers_path):
      src = os.path.join(local_explainers_path, item)
      dst = os.path.join(EXPLAINERS_DRIVE_DIR, item)
      if os.path.isdir(src):
          shutil.copytree(src, dst, dirs_exist_ok=True)
          shutil.rmtree(src)
      else:
          shutil.copy2(src, dst)
          os.remove(src)
    os.rmdir(local_explainers_path)
    os.symlink(EXPLAINERS_DRIVE_DIR, local_explainers_path)
    print('Migrated existing local explainers to Drive and symlinked.')
else:
    os.symlink(EXPLAINERS_DRIVE_DIR, local_explainers_path)
    print('Created Explainers -> Drive symlink.')

# Ensure HAM10000_metadata.csv is present and not empty
HAM_METADATA_PATH = f'{DATA_DIR}/HAM10000_metadata.csv'
if not os.path.exists(HAM_METADATA_PATH) or os.path.getsize(HAM_METADATA_PATH) == 0:
    print('Copying dataset from Drive...')
    # Ensure DATA_DIR is clean of previous HAM10000 data before copying
    !rm -f {DATA_DIR}/*.jpg
    !rm -f {HAM_METADATA_PATH}
    !cp {DRIVE_DIR}/*.jpg {DATA_DIR}/
    !cp {DRIVE_DIR}/HAM10000_metadata.csv {DATA_DIR}/
    print('Done.')
else:
    print('Already copied, skipping.')

meta = pd.read_csv(f'{DATA_DIR}/HAM10000_metadata.csv')
print(f"Total rows: {len(meta)}")
print(meta['dx'].value_counts())

import ICE.Data
import ICE.ModelWrapper
import ICE.Explainer
import ICE.utils

HAM_CLASSES = {
    'mel': 0,
    'nv': 1,
    'bcc': 2,
    'akiec': 3,
    'bkl': 4,
    'df': 5,
    'vasc': 6
}

meta = pd.read_csv(f'{DATA_DIR}/HAM10000_metadata.csv')
img_dir = Path(DATA_DIR)

# map each class to its corresponding images (and indices)
ham_dict = {}
for label_name, label_idx in HAM_CLASSES.items():
    paths = meta[meta['dx'] == label_name]['image_id'].apply(
        lambda x: str(img_dir / f"{x}.jpg")
    ).tolist()
    ham_dict[label_idx] = {
        'idx': label_idx,
        'name': label_name,
        'paths': paths
    }
for d in list(ham_dict.values()):
    print(d)

# define parameters
paras = {}
paras['target_classes'] = [HAM_CLASSES['mel']]
paras['classes_names'] = ['melanoma']
paras['n_components'] = 10
paras['layer_name'] = 'layer4'
paras['title'] = "ham_melanoma"
paras['overwrite'] = False

In [ ]:
import zipfile

DRIVE_ROOT    = '/content/drive/MyDrive/derm7pt'
DERM7PT_ZIP   = f'{DRIVE_ROOT}/release_v0.zip'
OUTPUT_DIR    =f'{DRIVE_ROOT}/outputs'
LOCAL_DATA  = '/content/derm7pt'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

# Check if it's already extracted correctly before doing anything else
matches = list(Path(LOCAL_DATA).rglob('meta.csv'))

if matches:
    print(f'Found existing extraction, skipping unzip.')
else:
    print('Not found — wiping and unzipping fresh...')
    shutil.rmtree(LOCAL_DATA, ignore_errors=True)
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(DERM7PT_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    print('Done.')
    matches = list(Path(LOCAL_DATA).rglob('meta.csv'))
    if not matches:
        raise FileNotFoundError('Could not find meta.csv even after unzipping — check DERM7PT_ZIP contents.')

LOCAL_DATA = str(matches[0].parent.parent)
print(f'Dataset root: {LOCAL_DATA}')

for required in ['meta/meta.csv', 'meta/train_indexes.csv',
                 'meta/valid_indexes.csv', 'meta/test_indexes.csv']:
    path = os.path.join(LOCAL_DATA, required)
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {status}  {required}')

CONFIG = {
    'dataset_root': LOCAL_DATA,
    'output_dir':   OUTPUT_DIR,
    'concepts': {
        'pigment_network':      {'labels': ['absent', 'typical', 'atypical'],  'n_classes': 3},
        'blue_whitish_veil':    {'labels': ['absent', 'present'],              'n_classes': 2},
        'vascular_structures':  {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3, 'label_mapping': {'arborizing': 'regular', 'comma': 'regular', 'hairpin': 'regular', 'within regression': 'regular', 'wreath': 'regular', 'dotted': 'irregular', 'linear irregular': 'irregular'}},
        'pigmentation':         {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3, 'label_mapping': {'diffuse regular': 'regular', 'localized regular': 'regular', 'diffuse irregular': 'irregular', 'localized irregular': 'irregular'}},
        'streaks':              {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3},
        'dots_and_globules':        {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3},
        'regression_structures':{'labels': ['absent', 'present'],              'n_classes': 2, 'label_mapping': {'blue areas': 'present', 'white areas': 'present', 'combinations': 'present',}},
    },
    'image_size':    224,
    'batch_size':    16,
    'num_epochs':    30,
    'learning_rate': 1e-4,
    'weight_decay':  1e-4,
    'cam_method':   'gradcam++',
    'device':        device,
}

print('Configuration set.')
print(f"  Dataset : {CONFIG['dataset_root']}")
print(f"  Outputs : {CONFIG['output_dir']}")
print(f"  CAM     : {CONFIG['cam_method']}")

In [ ]:
class Derm7ptDataset(Dataset):

    def __init__(self, dataset_root, split_indexes, concepts_config, transform=None):
        self.root = Path(dataset_root)
        self.transform = transform
        self.concepts_config = concepts_config
        df = pd.read_csv(self.root / 'meta' / 'meta.csv')
        self.df = df.iloc[split_indexes].reset_index(drop=True)
        self.label_encoders = {
            concept: {label: idx for idx, label in enumerate(cfg['labels'])}
            for concept, cfg in concepts_config.items()
        }
        self.precompute_numeric_labels()  # combining extra labels into known ones

    def precompute_numeric_labels(self):
        for concept, cfg in self.concepts_config.items():
            mapping = cfg.get('label_mapping', {})
            raw = self.df[concept].astype(str).str.lower().str.strip()
            raw = raw.map(lambda x: mapping.get(x, x))
            self.df[f'{concept}_numeric'] = raw.map(self.label_encoders[concept]).fillna(-1).astype(int)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root / 'images' / row['derm']
        image = Image.open(img_path).convert('RGB')
        w, h = image.size  # these 3 lines were added to crop the border, this is present in the derm7pt code
        crop_amount = 25
        image = image.crop((crop_amount, crop_amount, w - crop_amount, h - crop_amount))
        if self.transform:
            image = self.transform(image)
        labels = {concept: int(row[f'{concept}_numeric']) for concept in self.concepts_config}
        return image, labels, row['derm']


def load_split_indexes(dataset_root, split_name):
    path = Path(dataset_root) / 'meta' / f'{split_name}_indexes.csv'
    return list(pd.read_csv(path)['indexes'])


def get_transforms(image_size, split):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def collate_fn(batch):
    return (
        torch.stack([b[0] for b in batch]),
        [b[1] for b in batch],
        [b[2] for b in batch],
    )


# Load splits
cfg = CONFIG
train_idx = load_split_indexes(cfg['dataset_root'], 'train')
valid_idx = load_split_indexes(cfg['dataset_root'], 'valid')
test_idx  = load_split_indexes(cfg['dataset_root'], 'test')

train_dataset = Derm7ptDataset(cfg['dataset_root'], train_idx, cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'train'))
val_dataset   = Derm7ptDataset(cfg['dataset_root'], valid_idx, cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'valid'))
test_dataset  = Derm7ptDataset(cfg['dataset_root'], test_idx,  cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'test'))
for concept, ccfg in cfg['concepts'].items():
    mapping = ccfg.get('label_mapping', {})
    raw_vals = set(train_dataset.df[concept].astype(str).str.lower().str.strip().unique())
    mapped_vals = {mapping.get(v, v) for v in raw_vals}
    known = set(cfg['concepts'][concept]['labels'])
    unknown = mapped_vals - known
    print(f"{concept:25s} known={known}  unmapped={unknown}")

derm_train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
derm_val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### Create ICE explainer functions

These three functions are important to how the ICE explainer works, and will streamline work down the track.

In [ ]:
def make_imgs(label_idx):
  '''
  Makes returns a list of all image paths for the class specified by label_idx.
  '''
  return [(p, label_idx) for p in ham_dict[label_idx]['paths']]

def get_loaders(target_classes = []):
  '''
  Returns a list of data loaders for each class specified by target_classes.

  ICE needs separate loaders for each class to learn class-specific concepts.
  '''
  loaders = []
  for i,idx in enumerate(target_classes):
    loaders.append(DataLoader(
            ICE.Data.ImageDataset(make_imgs(idx),
                    transforms.Compose([
                    transforms.Resize((224,224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                            std=[0.229, 0.224, 0.225]),
                    ])),
            batch_size = 16,
            num_workers=2,
            #shuffle=True
    ))
  return loaders


def get_explainer(paras, model):
  classes_names = paras['classes_names']
  target_classes = paras['target_classes']
  n_components = paras['n_components'] # number of concepts to generate
  layer_name = paras['layer_name']
  title = paras['title']
  overwrite = paras['overwrite']

  # If overwriting, delete the old folder
  if overwrite:
    try:
        shutil.rmtree('Explainers/'+title)
    except:
        pass

  # Just predict for the specific target classes
  model.set_predict_target(target_classes)
  loaders = get_loaders(target_classes)

  # load the data if it exists already
  if os.path.exists(Path('Explainers')/title):
    Exp = ICE.Explainer.Explainer(title = title)
    Exp.load()
    print ('model loaded')
  else:
    # create an Explainer
    Exp = ICE.Explainer.Explainer(title = title,
                    layer_name = layer_name,
                    class_names = classes_names,
                    utils = ICE.utils.img_utils(mode = "torch"),
                    n_components = n_components,
                    reducer_type = "NMF"
                  )

    # turn each loader from the full image set to a subset of 1000 (per class)

    loaders = [DataLoader(
      Subset(loader.dataset, range(min(1000, len(loader.dataset)))),
      batch_size=16,
      num_workers=2
    ) for loader in loaders]

    # train reducer based on target classes
    Exp.train_model(model,loaders)
    # generate features
    Exp.generate_features(model, loaders)
    # save the explainer, use load to load it with the same title
    Exp.save()

  return Exp

In [ ]:
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

# Set random seed for reproducible splitting
np.random.seed(0)

# 1. Gather paths and labels for Melanoma and Nevus classes
mel_paths = [(p, 0) for p in ham_dict[HAM_CLASSES['mel']]['paths']]
nv_paths = [(p, 1) for p in ham_dict[HAM_CLASSES['nv']]['paths']]

# 2. Programmatically verify class sizes
# Expected: Melanoma = 445 + 223 + 445 = 1113; Nevus = 2682 + 1340 + 2683 = 6705
assert len(mel_paths) == 1113, f"Expected 1113 melanoma images, found {len(mel_paths)}"
assert len(nv_paths) == 6705, f"Expected 6705 nevus images, found {len(nv_paths)}"

mel_train_p, mel_temp_p = train_test_split(mel_paths, train_size=445, test_size=223+445, random_state=0)
nv_train_p, nv_temp_p = train_test_split(nv_paths, train_size=2682, test_size=1340+2683, random_state=0)
mel_val_p, mel_test_p = train_test_split(mel_temp_p, train_size=223, test_size=445, random_state=0)
nv_val_p, nv_test_p = train_test_split(nv_temp_p, train_size=1340, test_size=2683, random_state=0)

train_data_list = mel_train_p + nv_train_p
val_data_list = mel_val_p + nv_val_p
test_data_list = mel_test_p + nv_test_p

ham_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
ham_binary_dataset = ICE.Data.ImageDataset(train_data_list + val_data_list + test_data_list, ham_transforms)

total_train = len(train_data_list)
total_val = len(val_data_list)
total_test = len(test_data_list)

train_indices = list(range(0, total_train))
val_indices = list(range(total_train, total_train + total_val))
test_indices = list(range(total_train + total_val, total_train + total_val + total_test))

ham_train_subset = Subset(ham_binary_dataset, train_indices)
ham_val_subset = Subset(ham_binary_dataset, val_indices)
ham_test_subset = Subset(ham_binary_dataset, test_indices)

print(f"HAM10000 binary subsets created successfully:")
print(f"  Train: {len(ham_train_subset)} (Mel: {len(mel_train_p)}, Nev: {len(nv_train_p)})")
print(f"  Val:   {len(ham_val_subset)} (Mel: {len(mel_val_p)}, Nev: {len(nv_val_p)})")
print(f"  Test:  {len(ham_test_subset)} (Mel: {len(mel_test_p)}, Nev: {len(nv_test_p)})")

### Train Derm7pt image -> concept classifier

Taking a pretrained ResNet50 model and giving it the capability to predict all 7 Derm7pt concepts (and computing class weights)

In [ ]:
class Derm7ptMultiTaskModel(nn.Module):
    def __init__(self, concepts_config, dropout_rate=0.5, concept_channels=256):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4, # <- remove for better resolution
        )
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout_rate)
        self.concept_convs   = nn.ModuleDict({
            concept: nn.Conv2d(2048, concept_channels, kernel_size=1)  # was 1024 (for layer 4 exclusion)
            for concept, cfg in concepts_config.items()
        })

        self.heads = nn.ModuleDict({
            concept: nn.Linear(concept_channels, cfg['n_classes'])
            for concept, cfg in concepts_config.items()
        })

    def forward(self, x):
        feat   = self.features(x)
        # pooled = self.dropout(self.pool(feat).flatten(1))
        # return {concept: head(pooled) for concept, head in self.heads.items()}
        outputs = {}  # this instead does the forwards pass for each specific concept
        for concept, conv in self.concept_convs.items():
            c_feat = conv(feat)
            pooled = self.dropout(self.pool(c_feat).flatten(1))
            outputs[concept] = self.heads[concept](pooled)
        return outputs

def compute_concept_class_weights(dataset, concepts_config, device):  # fix this????
    all_labels = {c: [] for c in concepts_config}
    for _, label_dict, _ in dataset:
        for concept, label_idx in label_dict.items():
            if label_idx >= 0:
                all_labels[concept].append(label_idx)
    weights = {}
    for concept, cfg in concepts_config.items():
        labels_arr = np.array(all_labels[concept])
        present_classes    = np.unique(labels_arr)  # some small classes might not be in training set
        cw_present = compute_class_weight('balanced', classes=present_classes, y=labels_arr)

        mean_weight = cw_present.mean()
        cw = np.full(cfg['n_classes'], mean_weight, dtype=np.float64)
        for cls, w in zip(present_classes, cw_present):
            cw[cls] = w

        weights[concept] = torch.tensor(cw, dtype=torch.float).to(device)
        print(f'  {concept:30s}: {cw.round(2)}')
    return weights

derm_model = Derm7ptMultiTaskModel(cfg['concepts']).to(device)
print('Model built: ResNet-50 backbone + 7 concept heads')
total_params = sum(p.numel() for p in derm_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

print('Computing class weights...')
derm_class_weights = compute_concept_class_weights(train_dataset, cfg['concepts'], device)

### Training HAM10000 Model and ICE Explainer

In [ ]:
NUM_CLASSES = 7

m = models.resnet50(pretrained=True)
m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
m = m.cuda()

# Getting a map of labels and classes over the training subset
train_indices_set = set(train_indices)
all_imgs = []
all_labels = []
for idx in HAM_CLASSES.values():
    imgs = make_imgs(idx)

train_imgs = [ham_binary_dataset[i] for i in train_indices]
train_labels = [label for _, label in train_imgs]

# Weighted bias based on training class distribution
class_counts = [train_labels.count(i) for i in range(2)]
ham_class_weights = torch.tensor([1.0 / max(c, 1) for c in class_counts], dtype=torch.float).cuda()
ham_criterion = nn.CrossEntropyLoss(weight=ham_class_weights)

ham_optimizer = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)

train_loader = DataLoader(
    ham_train_subset,
    batch_size=16,
    num_workers=2,
    shuffle=True
)

val_loader = DataLoader(
    ham_val_subset,
    batch_size=16,
    num_workers=2,
    shuffle=False
)

NUM_EPOCHS = 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(ham_optimizer, T_max=NUM_EPOCHS)
HAM_CHECKPOINT = os.path.join(OUTPUT_DIR, 'ham_resnet50.pth')

if os.path.exists(HAM_CHECKPOINT):
    m.load_state_dict(torch.load(HAM_CHECKPOINT, map_location=device))
    print('Loaded existing HAM10000 classifier.')
else:
    m.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.permute(0, 3, 1, 2).cuda(), labels.cuda()
            ham_optimizer.zero_grad()
            outputs = m(inputs)

            restricted_outputs = outputs[:, [HAM_CLASSES['mel'], HAM_CLASSES['nv']]]
            loss = ham_criterion(restricted_outputs, labels)
            loss.backward()
            ham_optimizer.step()
            running_loss += loss.item()
            _, predicted = restricted_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()

        m.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for val_inputs, val_labels in val_loader:
                val_inputs, val_labels = val_inputs.permute(0, 3, 1, 2).cuda(), val_labels.cuda()
                val_outputs = m(val_inputs)
                val_restricted = val_outputs[:, [HAM_CLASSES['mel'], HAM_CLASSES['nv']]]
                _, val_predicted = val_restricted.max(1)
                val_total += val_labels.size(0)
                val_correct += val_predicted.eq(val_labels).sum().item()
        m.train()

        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Train Acc: {100.*correct/total:.1f}%, Val Acc: {100.*val_correct/val_total:.1f}%")

    torch.save(m.state_dict(), HAM_CHECKPOINT)

m.eval()
ham_model = ICE.ModelWrapper.PytorchModelWrapper(
    m,
    batch_size=16,
    predict_target=paras['target_classes'],
    input_channel_first=False,
    model_channel_first=True
)

In [ ]:
# Two-class explainer: melanoma vs nevi
target_classes = [HAM_CLASSES['mel'], HAM_CLASSES['nv']]
classes_names = ['melanoma', 'nevi']
title = "ham_mel_vs_nv"
n_components = 10
layer_name = 'layer4'
overwrite = True

paras = {}
paras['target_classes'] = target_classes
paras['classes_names'] = classes_names
paras['n_components'] = n_components
paras['layer_name'] = layer_name
paras['title'] = title
paras['overwrite'] = overwrite

Exp = get_explainer(paras, ham_model)

fidelity = Exp.reducer_err
for name, err in zip(classes_names, fidelity):
    print(f"Fidelity error ({name}): {err:.4f} ({err*100:.2f}%)")

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import shutil

mel_idx = HAM_CLASSES['mel']
nv_idx  = HAM_CLASSES['nv']

@torch.no_grad()
def collect_binary_mel_nv_preds(model, loader, mel_idx, nv_idx, device):
    model.eval()
    all_preds = []
    all_labels = []
    for inputs, labels in loader:
        inputs = inputs.permute(0, 3, 1, 2).to(device)
        outputs = model(inputs)
        # restrict to just the two target classes, matching set_predict_target's framing
        restricted = outputs[:, [mel_idx, nv_idx]]
        binary_pred = restricted.argmax(dim=1)  # 0 -> mel, 1 -> nv
        all_preds.extend(binary_pred.cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_preds), np.array(all_labels)

# Calculate metrics on the clean validation loader
preds, labels = collect_binary_mel_nv_preds(m, val_loader, mel_idx, nv_idx, device)

# only keep rows that are actually mel or nv — restricted argmax is meaningless for other classes
mask = np.isin(labels, [mel_idx, nv_idx])
binary_true = (labels[mask] == nv_idx).astype(int)   # 0=mel, 1=nv
binary_pred = preds[mask]

acc = accuracy_score(binary_true, binary_pred)
f1  = f1_score(binary_true, binary_pred, average='macro', zero_division=0)
print(f"Restricted mel-vs-nv (explainer framing) on Validation Subset — Accuracy: {acc*100:.2f}%  F1: {f1:.3f}")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

# Calculate per-class accuracy

# Melanoma (where binary_true is 0)
mel_mask = (binary_true == 0)
mel_true = binary_true[mel_mask]
mel_pred = binary_pred[mel_mask]
mel_f1 = f1_score(mel_true, mel_pred, average='binary', pos_label=0, zero_division=0)

# Nevus (where binary_true is 1)
nv_mask = (binary_true == 1)
nv_true = binary_true[nv_mask]
nv_pred = binary_pred[nv_mask]
nv_f1 = f1_score(nv_true, nv_pred, average='binary', pos_label=1, zero_division=0)

# print(f"Melanoma (explainer framing) Accuracy: {mel_acc*100:.2f}%  F1: {mel_f1:.3f}")
# print(f"Nevus (explainer framing) Accuracy: {nv_acc*100:.2f}%  F1: {nv_f1:.3f}")

### Training Derm7pt Model and per-concept GradCAM

In [ ]:
def train_one_epoch(model, loader, optimizer, class_weights, device):
    model.train()
    total_loss = 0.0

    concept_loss_totals = {c: 0.0 for c in class_weights}
    concept_loss_counts = {c: 0 for c in class_weights}

    for images, label_dicts, _ in loader:
        images = images.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        concept_losses = {}
        for concept, logits in outputs.items():
            labels = torch.tensor(
                [d[concept] for d in label_dicts], dtype=torch.long
            ).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            criterion = nn.CrossEntropyLoss(weight=class_weights[concept], ignore_index=-1)
            concept_losses[concept] = criterion(logits[valid], labels[valid])

        batch_loss = sum(concept_losses.values())
        batch_loss.backward()
        optimizer.step()
        total_loss += batch_loss.item()
        for concept, loss_val in concept_losses.items():
            concept_loss_totals[concept] += loss_val.item()
            concept_loss_counts[concept] += 1

    avg_concept_losses = {
        c: (concept_loss_totals[c] / concept_loss_counts[c] if concept_loss_counts[c] > 0 else float('nan'))
        for c in class_weights
    }
    return total_loss / len(loader), avg_concept_losses


@torch.no_grad()
def evaluate(model, loader, class_weights, device):
    model.eval()
    total_loss = 0.0
    correct = {c: 0 for c in cfg['concepts']}
    counts  = {c: 0 for c in cfg['concepts']}
    for images, label_dicts, _ in loader:
        images  = images.to(device)
        outputs = model(images)
        for concept, logits in outputs.items():
            labels = torch.tensor(
                [d[concept] for d in label_dicts], dtype=torch.long
            ).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            criterion = nn.CrossEntropyLoss(weight=class_weights[concept], ignore_index=-1)
            total_loss += criterion(logits[valid], labels[valid]).item()
            preds = logits[valid].argmax(dim=1)
            correct[concept] += (preds == labels[valid]).sum().item()
            counts[concept]  += valid.sum().item()
    accs = {c: (correct[c] / counts[c] if counts[c] > 0 else 0.0) for c in correct}
    return total_loss / len(loader), accs # check if this is good for per-class

def train_derm_model(cfg, seed, checkpoint_path, device):
    derm_model = Derm7ptMultiTaskModel(cfg['concepts']).to(device)

    if os.path.exists(checkpoint_path):
        print(f'Found existing checkpoint at {checkpoint_path} — loading.')
        derm_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        return derm_model

    derm_optimizer = optim.Adam(derm_model.parameters(),
                           lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(derm_optimizer, T_max=cfg['num_epochs']) # should this be different?
    best_val_loss = float('inf')

    concept_loss_history = {c: [] for c in cfg['concepts']}

    for epoch in range(cfg['num_epochs']):
        train_loss, train_concept_losses = train_one_epoch(derm_model, derm_train_loader, derm_optimizer, derm_class_weights, device)
        val_loss, val_accs = evaluate(derm_model, derm_val_loader, derm_class_weights, device)
        scheduler.step() # what does this do?

        for concept, loss_val in train_concept_losses.items():
            concept_loss_history[concept].append(loss_val)

        print(f'[seed={seed}] Epoch {epoch+1:02d}/{cfg["num_epochs"]} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f}')
        for concept in train_concept_losses:
            print(f'   {concept:30s}: loss={train_concept_losses[concept]:.4f}  acc={val_accs[concept]*100:.1f}%')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(derm_model.state_dict(), checkpoint_path)
            print(f'   ✓ Saved (val_loss={val_loss:.4f}) → {checkpoint_path}')

    plt.figure(figsize=(10, 6))
    for concept, losses in concept_loss_history.items():
        plt.plot(losses, label=concept)
    plt.xlabel('Epoch')
    plt.ylabel('Train Loss')
    plt.legend(fontsize=8)
    plt.title(f'Per-Concept Training Loss (seed={seed})')
    plt.savefig(os.path.join(OUTPUT_DIR, f'concept_loss_curves_seed{seed}.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Training complete for seed={seed}.')
    return derm_model

N_RUNS = 5
SEEDS = [0, 1, 2, 3, 4]

DERM_MODEL_CHECKPOINTS = [
    os.path.join(OUTPUT_DIR, f'best_model_run{i+1}.pth') for i in range(N_RUNS)
]

for seed, checkpoint_path in zip(SEEDS, DERM_MODEL_CHECKPOINTS):
    print(f"\n=== Run {seed+1}/{N_RUNS} (seed={seed}) ===")
    train_derm_model(cfg, seed, checkpoint_path, device)

In [ ]:
MELANOMA_DIAGNOSES = {  # taken from printing the diagnoses in the Derm7pt data, then choosing the ones that were melanoma or nevus
    'melanoma': 'melanoma',
    'melanoma (in situ)': 'melanoma',
    'melanoma (less than 0.76 mm)': 'melanoma',
    'melanoma (0.76 to 1.5 mm)': 'melanoma',
    'melanoma (more than 1.5 mm)': 'melanoma',
    'melanoma metastasis': 'melanoma',
}
NEVUS_DIAGNOSES = {
    'clark nevus': 'nevus',
    'reed or spitz nevus': 'nevus',
    'blue nevus': 'nevus',
    'combined nevus': 'nevus',
    'congenital nevus': 'nevus',
    'dermal nevus': 'nevus',
    'recurrent nevus': 'nevus',
}

class FullFeatureBaseline(nn.Module):
    """Predicts each concept directly from pooled backbone features,
    with no concept-bottleneck compression, so we can measure how much
    predictive signal the bottleneck heads retain relative to this."""
    def __init__(self, concepts_config, backbone_channels=1024):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.heads = nn.ModuleDict({
            concept: nn.Linear(backbone_channels, cfg['n_classes'])
            for concept, cfg in concepts_config.items()
        })

    def forward(self, raw_feat):
        pooled = self.dropout(self.pool(raw_feat).flatten(1))
        return {concept: head(pooled) for concept, head in self.heads.items()}

def train_full_feature_baseline(derm_model, loader, class_weights, device, epochs=30, lr=1e-3):
    backbone_channels = 2048
    baseline = FullFeatureBaseline(cfg['concepts'], backbone_channels).to(device)

    BASELINE_CHECKPOINT = os.path.join(OUTPUT_DIR, 'baseline_model.pth')
    if os.path.exists(BASELINE_CHECKPOINT):
        baseline.load_state_dict(torch.load(BASELINE_CHECKPOINT, map_location=device))
        print('Loaded existing baseline model.')
        return baseline

    optimizer = optim.Adam(baseline.parameters(), lr=lr)
    derm_model.eval()  # freeze backbone — no grad flows into it

    for epoch in range(epochs):
        baseline.train()
        running_loss = 0.0
        for images, label_dicts, _ in loader:
            images = images.to(device)
            with torch.no_grad():
                raw_feat = derm_model.features(images)

            optimizer.zero_grad()
            outputs = baseline(raw_feat)
            losses = []
            for concept, logits in outputs.items():
                labels = torch.tensor([d[concept] for d in label_dicts], dtype=torch.long).to(device)
                valid = labels >= 0
                if valid.sum() == 0:
                    continue
                criterion = nn.CrossEntropyLoss(weight=class_weights[concept], ignore_index=-1)
                losses.append(criterion(logits[valid], labels[valid]))
            loss = sum(losses)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"  Baseline epoch {epoch+1}/{epochs} | loss: {running_loss/len(loader):.4f}")

    torch.save(baseline.state_dict(), BASELINE_CHECKPOINT)
    return baseline

@torch.no_grad()
def evaluate_baseline(derm_model, baseline, loader, device):
    derm_model.eval()
    baseline.eval()
    correct = {c: 0 for c in cfg['concepts']}
    counts  = {c: 0 for c in cfg['concepts']}
    for images, label_dicts, _ in loader:
        images = images.to(device)
        raw_feat = derm_model.features(images)
        outputs = baseline(raw_feat)
        for concept, logits in outputs.items():
            labels = torch.tensor([d[concept] for d in label_dicts], dtype=torch.long).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            preds = logits[valid].argmax(dim=1)
            correct[concept] += (preds == labels[valid]).sum().item()
            counts[concept]  += valid.sum().item()
    return {c: (correct[c] / counts[c] if counts[c] > 0 else 0.0) for c in correct}

def filter_dataset_by_diagnosis(dataset, diagnosis_set, label):
    diag = dataset.df['diagnosis'].astype(str).str.lower().str.strip()
    mask = diag.isin(diagnosis_set)
    indices = dataset.df.index[mask].tolist()
    print(f"{label:10s}: {len(indices)} samples matched")
    return Subset(dataset, indices)

val_melanoma = filter_dataset_by_diagnosis(val_dataset, MELANOMA_DIAGNOSES, "Melanoma")
val_nevus    = filter_dataset_by_diagnosis(val_dataset, NEVUS_DIAGNOSES, "Nevus")

melanoma_loader = DataLoader(val_melanoma, batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=collate_fn)
nevus_loader    = DataLoader(val_nevus, batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=collate_fn)

baseline_model = train_full_feature_baseline(derm_model, derm_train_loader, derm_class_weights, device, epochs=30)

def compute_avg_fidelity(derm_model, baseline_model, loader, class_weights, device, label=""):
    baseline_accs = evaluate_baseline(derm_model, baseline_model, loader, device)
    _, bottleneck_accs = evaluate(derm_model, loader, class_weights, device)

    baseline_acc = []
    bottleneck_acc = []
    for concept in cfg['concepts']:
        baseline_acc.append(baseline_accs[concept])
        bottleneck_acc.append(bottleneck_accs[concept])

    baseline_acc = np.array(baseline_acc)
    bottleneck_acc = np.array(bottleneck_acc)
    epsilon = 1e-4  # same as ICE paper
    fid_err = abs(baseline_acc - bottleneck_acc).mean(axis=0) / (abs(baseline_acc).mean(axis=0) + epsilon)
    print(f"{label} — average fidelity error across concepts: {fid_err*100:.2f}%, accuracy: {abs(baseline_acc).mean(axis=0)}")
    return fid_err

mel_fidelity   = compute_avg_fidelity(derm_model, baseline_model, melanoma_loader, derm_class_weights, device, label="Melanoma")
nevus_fidelity = compute_avg_fidelity(derm_model, baseline_model, nevus_loader, derm_class_weights, device, label="Nevus")

In [ ]:
@torch.no_grad()
def collect_concept_preds_labels(model, loader, device):
    model.eval()
    all_preds = {c: [] for c in cfg['concepts']}
    all_labels = {c: [] for c in cfg['concepts']}
    for images, label_dicts, _ in loader:
        images = images.to(device)
        outputs = model(images)
        for concept, logits in outputs.items():
            labels = torch.tensor([d[concept] for d in label_dicts], dtype=torch.long).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            preds = logits[valid].argmax(dim=1).cpu().numpy()
            all_preds[concept].extend(preds.tolist())
            all_labels[concept].extend(labels[valid].cpu().numpy().tolist())
    return all_preds, all_labels

def compute_avg_accuracy_and_f1(model, loader, label=""):
    preds, labels = collect_concept_preds_labels(model, loader, device)

    accs, f1s = [], []
    for concept in cfg['concepts']:
        if len(labels[concept]) == 0:
            continue
        acc = accuracy_score(labels[concept], preds[concept])
        f1  = f1_score(labels[concept], preds[concept], average='macro', zero_division=0)
        accs.append(acc)
        f1s.append(f1)

    avg_acc = np.mean(accs)
    avg_f1  = np.mean(f1s)
    print(f"{label} — Accuracy: {avg_acc*100:.2f}%  F1: {avg_f1:.3f}  (averaged over {len(accs)} concepts)")
    return avg_acc, avg_f1

mel_acc_derm, mel_f1_derm = compute_avg_accuracy_and_f1(derm_model, melanoma_loader, label="Melanoma (derm7pt bottleneck)")
nev_acc_derm, nev_f1_derm = compute_avg_accuracy_and_f1(derm_model, nevus_loader, label="Nevus (derm7pt bottleneck)")

In [ ]:
class ConceptClassTarget:
    """Routes GradCAM gradients through a specific concept head and class."""
    def __init__(self, concept_name, class_idx):
        self.concept_name = concept_name
        self.class_idx    = class_idx
    def __call__(self, model_output):
        if model_output.dim() == 1:
            model_output = model_output.unsqueeze(0)
        return model_output[:, self.class_idx]

class SingleConceptWrapper(nn.Module):
    """Wraps the multi-task model to output only one concept's logits.
    GradCAM expects a plain tensor output, not a dict."""
    def __init__(self, model, concept_name):
        super().__init__()
        self.model        = model
        self.concept_name = concept_name
    def forward(self, x):
        return self.model(x)[self.concept_name]

def get_cam(method_name, model, target_layer):
    methods = {
        'gradcam':   GradCAM,
        'gradcam++': GradCAMPlusPlus,
        'scorecam':  ScoreCAM,
        'eigencam':  EigenCAM,
    }
    return methods[method_name.lower()](model=model, target_layers=[target_layer])

def suppress_cam_edges(cam, margin_frac=0.08, renormalise=True):
    h, w = cam.shape
    my, mx = int(h * margin_frac), int(w * margin_frac)
    mask = np.ones_like(cam)
    for i in range(my):
        mask[i, :] *= i / my
        mask[-i-1, :] *= i / my
    for j in range(mx):
        mask[:, j] *= j / mx
        mask[:, -j-1] *= j / mx
    cam_masked = cam * mask
    if renormalise and cam_masked.max() > 0:
        cam_masked = cam_masked / cam_masked.max()
    return cam_masked

def generate_concept_cams(model, image_tensor, concepts_config, cam_method_name, device):
    model.eval()
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        outputs = model(image_tensor)
    predicted_classes = {
        concept: logits.argmax(dim=1).item()
        for concept, logits in outputs.items()
    }
    concept_cams = {}

    for concept in concepts_config:
        pred_class = predicted_classes[concept]
        wrapper = SingleConceptWrapper(model, concept)
        wrapper.eval()

        # target_layer = wrapper.model.features[-2]
        target_layer = wrapper.model.concept_convs[concept] # changing it to per-concept  target layer

        cam = get_cam(cam_method_name, wrapper, target_layer)
        grayscale_cam = cam(
            input_tensor=image_tensor,
            targets=[ConceptClassTarget(concept, pred_class)]
            # eigen_smooth=True,  # both smooths have been added
            # aug_smooth=True
        )
        cam_suppressed = suppress_cam_edges(grayscale_cam[0], margin_frac=0.08, renormalise=True)
        concept_cams[concept] = cam_suppressed

    return concept_cams, predicted_classes

print('GradCAM functions defined.')

### Extracting CAVs and comparing mappings

Qualifying CAV similarity with cosine similarity equation, and using the hungarian algorithm for the one-to-one mapping constraint

In [ ]:
import cv2

def load_ham_image_array(ham_image_id):
    ham_img_path = Path(DATA_DIR) / f"{ham_image_id}.jpg"
    image = Image.open(ham_img_path).convert('RGB')
    image = transforms.Resize((224, 224))(image)
    image = transforms.ToTensor()(image)
    image = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(image).numpy()
    return np.transpose(image, (1, 2, 0))

def load_derm_image_tensor(derm_filename, cfg, transform):
  derm_img_path = Path(cfg['dataset_root']) / 'images' / derm_filename
  image = Image.open(derm_img_path).convert('RGB')
  w, h = image.size
  crop_amount = 25
  image = image.crop((crop_amount, crop_amount, w - crop_amount, h - crop_amount))
  return transform(image).unsqueeze(0)

def get_ham_cavs(x, model, Exp):
    raw_feat = model.get_feature(x, Exp.layer_name) # gets raw layer 4 activations
    h = Exp.reducer.transform(raw_feat)[0]   # NMF-projected -> shape (1, H, W, n_components)
    return {f'ham_concept_{k}': h[:, :, k] for k in range(Exp.n_components)}  # (H, W, n_components) for this one image

def get_derm_cavs(x, model):
  # Generate per-concept CAMs
  cavs, predicted_classes = generate_concept_cams(
    model, x, cfg['concepts'], cfg['cam_method'], device
  )
  return cavs

def cosine_sim(a, b):
    a, b = a.flatten(), b.flatten()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else None

def cross_model_similarity_matrix(derm_cavs: dict, ham_cavs: dict):
    derm_names = list(derm_cavs.keys())
    ham_names = list(ham_cavs.keys())
    sample_derm = list(derm_cavs.values())[0]
    sample_ham  = list(ham_cavs.values())[0]
    if sample_derm.shape != sample_ham.shape:
        target_h, target_w = sample_ham.shape
        derm_cavs = {
            k: cv2.resize(v.astype(np.float32), (target_w, target_h), interpolation=cv2.INTER_LINEAR)
            for k, v in derm_cavs.items()
        }

    matrix = np.zeros((len(derm_names), len(ham_names)))
    for i, dn in enumerate(derm_names):
        for j, hn in enumerate(ham_names):
            sim = cosine_sim(derm_cavs[dn], ham_cavs[hn])
            matrix[i, j] = sim if sim is not None else -1  # push undefined pairs to bottom of ranking
    return matrix, derm_names, ham_names

def report_best_matches(derm_cavs, ham_cavs):
    matrix, derm_names, ham_names = cross_model_similarity_matrix(derm_cavs, ham_cavs)
    row_ind, col_ind = linear_sum_assignment(-matrix)
    print("--- Best 1-to-1 matches (Hungarian assignment: monosemantic assumption) ---")
    for r, c in sorted(zip(row_ind, col_ind), key=lambda rc: -matrix[rc]):
        print(f"  {derm_names[r]:25s} <-> {ham_names[c]:15s} : cosine={matrix[r, c]:.3f}")

    print("\n--- Best match for each concept (not forced 1-to-1: polysemantic assumption) ---")
    for i, dn in enumerate(derm_names):
        best_j = matrix[i].argmax()
        print(f"  {dn:25s} best match: {ham_names[best_j]:15s} cosine={matrix[i, best_j]:.3f}")

    return matrix

In [ ]:
import shutil

def run_matching_for_model(derm_model, cfg, test_transform, ham_model, Exp):
    melanoma_results = {}
    nevus_results = {}
    RESULTS = {'melanoma': melanoma_results, 'nevus': nevus_results}
    CSV_PATHS = {
        'melanoma': os.path.join(OUTPUT_DIR, 'matched_pairs_melanoma_TOP100.csv'),
        'nevus': os.path.join(OUTPUT_DIR, 'matched_pairs_nevus_TOP100.csv'),
    }

    for label, csv_path in CSV_PATHS.items():
        if not os.path.exists(csv_path):
            print(f'Skipping {label}, file not found at {csv_path}...')
            continue
        table = pd.read_csv(csv_path)
        print(f"\n=== {label} ({16} pairs) ===")
        results_dict = RESULTS[label]

        for _, row in table.head(16).iterrows():
            derm_filename = row['derm7pt_file']
            ham_id = row['ham10000_id']
            print(f"\n-- pair: {derm_filename}  <->  {ham_id}  [{label}]  (csv cosine={row.get('cosine_similarity', float('nan')):.4f}) --")
            local_ham_path = os.path.join(DATA_DIR, f"{ham_id}.jpg")
            if not os.path.exists(local_ham_path):
                drive_ham_path = os.path.join(DRIVE_DIR, f"{ham_id}.jpg")
                if os.path.exists(drive_ham_path):
                    print(f"    [Auto-recovery] Copying missing image {ham_id}.jpg from Drive...")
                    shutil.copy2(drive_ham_path, local_ham_path)
                else:
                    print(f"    [Warning] Image {ham_id}.jpg could not be found on Drive either.")

            derm_img_tensor = load_derm_image_tensor(derm_filename, cfg, test_transform)
            derm_cavs = get_derm_cavs(derm_img_tensor, derm_model)
            ham_img_array = load_ham_image_array(ham_id)
            ham_cavs = get_ham_cavs(ham_img_array, ham_model, Exp)
            matrix = report_best_matches(derm_cavs, ham_cavs)
            results_dict[(derm_filename, ham_id)] = matrix

    return melanoma_results, nevus_results


test_transform = get_transforms(cfg['image_size'], 'test')

DERM_MODEL_CHECKPOINTS = [
    os.path.join(OUTPUT_DIR, 'best_model_run1.pth'),
    os.path.join(OUTPUT_DIR, 'best_model_run2.pth'),
    os.path.join(OUTPUT_DIR, 'best_model_run3.pth'),
    os.path.join(OUTPUT_DIR, 'best_model_run4.pth'),
    os.path.join(OUTPUT_DIR, 'best_model_run5.pth'),
]

mel_matrices_per_run = []
nev_matrices_per_run = []

for ckpt_path in DERM_MODEL_CHECKPOINTS:
    print(f"\n=== Loading derm model checkpoint: {ckpt_path} ===")
    if not os.path.exists(ckpt_path):
        print(f"Skipping missing checkpoint {ckpt_path}...")
        continue
    derm_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    derm_model.eval()

    melanoma_results, nevus_results = run_matching_for_model(
        derm_model, cfg, test_transform, ham_model, Exp
    )

    if len(melanoma_results) > 0:
        mel_matrices_per_run.append(np.stack(list(melanoma_results.values())))
    if len(nevus_results) > 0:
        nev_matrices_per_run.append(np.stack(list(nevus_results.values())))


### Visualising results

(This code is AI-generated)

In [ ]:
# ---------------------------------------------------
# Aggregation
# ---------------------------------------------------

def get_matrix_stack(results_dict):
    return np.stack(list(results_dict.values()))


def aggregate_stats(matrices, derm_names, ham_names):
    n_pairs = matrices.shape[0]
    mean_matrix = matrices.mean(axis=0)
    std_matrix  = matrices.std(axis=0)
    best_avg_idx = mean_matrix.argmax(axis=1)
    per_image_best = matrices.argmax(axis=2)

    stats = {'n_pairs': n_pairs, 'mean_matrix': mean_matrix, 'std_matrix': std_matrix, 'per_concept': {}}
    for i, dn in enumerate(derm_names):
        counts = np.bincount(per_image_best[:, i], minlength=len(ham_names))
        freq_winner_idx = counts.argmax()
        stats['per_concept'][dn] = {
            'best_avg_match': ham_names[freq_winner_idx],
            'best_avg_score': mean_matrix[i, freq_winner_idx],
            'best_avg_std': std_matrix[i, freq_winner_idx],
            'most_frequent_match': ham_names[freq_winner_idx],
            'most_frequent_pct': counts[freq_winner_idx] / n_pairs,
        }
    return stats


def aggregate_stats_run1_with_run_error(matrices_per_run, derm_names, ham_names):
    per_run_mean = np.stack([m.mean(axis=0) for m in matrices_per_run])  # (n_runs, n_derm, n_ham)
    run1_matrix = per_run_mean[0]
    best_idx = run1_matrix.argmax(axis=1)

    stats = {'per_concept': {}}
    for i, dn in enumerate(derm_names):
        j = best_idx[i]
        stats['per_concept'][dn] = {
            'best_avg_match': ham_names[j],
            'best_avg_score': run1_matrix[i, j],
            'best_avg_std':   per_run_mean[:, i, j].std(),
        }
    return stats


def get_consistency_stats_with_run_std(matrices_per_run, derm_names, ham_names):
    # Calculate top matches and matching pct per run
    run_pcts = []
    for run_m in matrices_per_run:
        per_image_best = run_m.argmax(axis=2)  # (n_pairs, n_concepts)
        n_pairs = run_m.shape[0]
        run_concept_pcts = {}
        for i, dn in enumerate(derm_names):
            counts = np.bincount(per_image_best[:, i], minlength=len(ham_names))
            freq_winner_idx = counts.argmax()
            run_concept_pcts[dn] = (counts[freq_winner_idx] / n_pairs) * 100
        run_pcts.append(run_concept_pcts)

    # Get average and standard deviation over runs
    stats = {'per_concept': {}}
    for dn in derm_names:
        pct_values = [run[dn] for run in run_pcts]
        stats['per_concept'][dn] = {
            'mean_pct': np.mean(pct_values),
            'std_pct': np.std(pct_values)
        }
    return stats


derm_names = list(cfg['concepts'].keys())
ham_names  = [f'ham_concept_{k}' for k in range(Exp.n_components)]

mel_matrices = np.concatenate(mel_matrices_per_run, axis=0)
nev_matrices = np.concatenate(nev_matrices_per_run, axis=0)
combined_matrices = np.concatenate([mel_matrices, nev_matrices], axis=0)

mel_stats      = aggregate_stats(mel_matrices, derm_names, ham_names)
nev_stats      = aggregate_stats(nev_matrices, derm_names, ham_names)
combined_stats = aggregate_stats(combined_matrices, derm_names, ham_names)

mel_stats_run_1 = aggregate_stats_run1_with_run_error(mel_matrices_per_run, derm_names, ham_names)
nev_stats_run_1 = aggregate_stats_run1_with_run_error(nev_matrices_per_run, derm_names, ham_names)


# ---------------------------------------------------
# Visualization 1: Heatmaps
# ---------------------------------------------------

def plot_heatmaps(mel_stats, nev_stats, derm_names, ham_names):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    mats = [mel_stats['mean_matrix'], nev_stats['mean_matrix'],
            mel_stats['mean_matrix'] - nev_stats['mean_matrix']]
    titles = ['Melanoma: mean cosine sim', 'Nevus: mean cosine sim', 'Difference (Mel − Nev)']
    cmaps = ['viridis', 'viridis', 'RdBu_r']

    for ax, mat, title, cmap in zip(axes, mats, titles, cmaps):
        vmax = np.abs(mat).max() if cmap == 'RdBu_r' else mat.max()
        vmin = -vmax if cmap == 'RdBu_r' else 0
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_xticks(range(len(ham_names)))
        ax.set_xticklabels(ham_names, rotation=90, fontsize=8)
        ax.set_yticks(range(len(derm_names)))
        ax.set_yticklabels(derm_names, fontsize=9)
        ax.set_title(title, fontsize=11)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center',
                         fontsize=6, color='white' if cmap == 'viridis' else 'black')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'concept_alignment_heatmaps.png'), bbox_inches='tight', dpi=150)
    plt.show()

plot_heatmaps(mel_stats, nev_stats, derm_names, ham_names)


# ---------------------------------------------------
# Visualization 2: Best Match Confidence
# ---------------------------------------------------

def plot_best_match_bars(mel_stats, nev_stats, derm_names):
    x = np.arange(len(derm_names))
    width = 0.35

    mel_scores = [mel_stats['per_concept'][dn]['best_avg_score'] for dn in derm_names]
    mel_stds   = [mel_stats['per_concept'][dn]['best_avg_std'] for dn in derm_names]
    nev_scores = [nev_stats['per_concept'][dn]['best_avg_score'] for dn in derm_names]
    nev_stds   = [nev_stats['per_concept'][dn]['best_avg_std'] for dn in derm_names]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width/2, mel_scores, width, yerr=mel_stds, capsize=3, label='Melanoma', color='#d62728')
    ax.bar(x + width/2, nev_scores, width, yerr=nev_stds, capsize=3, label='Nevus', color='#1f77b4')

    ax.set_xticks(x)
    ax.set_xticklabels(derm_names, rotation=30, ha='right')
    ax.set_ylabel('Best-match cosine similarity (mean  std)')
    ax.set_title('Strength of best ICE-concept match, per derm concept')
    ax.legend()
    ax.axhline(0, color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'best_match_scores_bar.png'), bbox_inches='tight', dpi=150)
    plt.show()

plot_best_match_bars(mel_stats_run_1, nev_stats_run_1, derm_names)


# ---------------------------------------------------
# Visualization 3: Stability / Consistency Plot with 5-Model Error Bars
# ---------------------------------------------------

def plot_match_consistency(mel_matrices_list, nev_matrices_per_run, derm_names, ham_names):
    mel_consistency = get_consistency_stats_with_run_std(mel_matrices_list, derm_names, ham_names)
    nev_consistency = get_consistency_stats_with_run_std(nev_matrices_per_run, derm_names, ham_names)

    x = np.arange(len(derm_names))
    width = 0.35

    mel_mean = [mel_consistency['per_concept'][dn]['mean_pct'] for dn in derm_names]
    mel_std  = [mel_consistency['per_concept'][dn]['std_pct'] for dn in derm_names]
    nev_mean = [nev_consistency['per_concept'][dn]['mean_pct'] for dn in derm_names]
    nev_std  = [nev_consistency['per_concept'][dn]['std_pct'] for dn in derm_names]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width/2, mel_mean, width, yerr=mel_std, capsize=3, label='Melanoma', color='#d62728')
    ax.bar(x + width/2, nev_mean, width, yerr=nev_std, capsize=3, label='Nevus', color='#1f77b4')

    ax.axhline(100 / len(ham_names), color='gray', linestyle='--', linewidth=1,
               label=f'Chance level ({100/len(ham_names):.0f}%)')

    ax.set_xticks(x)
    ax.set_xticklabels(derm_names, rotation=30, ha='right')
    ax.set_ylabel('% of images where this ICE concept was the top match')
    ax.set_title('Consistency of best-match ICE concept across 5 training runs')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'match_consistency_bar_with_errorbars.png'), bbox_inches='tight', dpi=150)
    plt.show()

plot_match_consistency(mel_matrices_per_run, nev_matrices_per_run, derm_names, ham_names)